In [ ]:
# MODEL: P01
# ONLY CHANGE: a0, b0 hierarchical (Half-Cauchy + MH)
# EVERYTHING ELSE KEPT AS YOUR WEEKLY VERSION, INCLUDING tqdm + display

# ================================================================
# Imports
# ================================================================
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, coo_matrix, diags, bmat
from scipy.sparse.csgraph import connected_components
from sksparse.cholmod import cholesky
from polyagamma import random_polyagamma
import geopandas as gpd
import pyreadr
import pickle
from IPython.display import clear_output, display

# ================================================================
# Load data (remove isolated points)
# ================================================================
no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

snow = pyreadr.read_r("snow_cleaned_full.Rda")
snow = list(snow.values())[0]

all_y = snow.drop(index=no_nbs).reset_index(drop=True)
coords_full = all_y.iloc[:, :2].to_numpy()
y_full      = all_y.iloc[:, 2:].to_numpy()

S_full, TT = y_full.shape
period = 52

# ================================================================
# Global time trend (scaled once)
# ================================================================
t_full = np.arange(1, TT + 1)
t_trend_full = (t_full - t_full.mean()) / t_full.std(ddof=0)

# ================================================================
# Build adjacency on FULL graph
# ================================================================
gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords_full[:, 0], coords_full[:, 1]),
    crs="EPSG:4326"
)
gdf = gdf.to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")
xy_full = np.vstack([gdf.geometry.x, gdf.geometry.y]).T / 1e6

A_full = (squareform(pdist(xy_full)) <= 0.22).astype(int)
np.fill_diagonal(A_full, 0)
A_full = csr_matrix(A_full)

# ================================================================
# Merge two largest connected components
# ================================================================
n_comp, labels = connected_components(A_full, directed=False)
sizes = np.bincount(labels)
comp1, comp2 = np.argsort(sizes)[-2:][::-1]

keep = np.where((labels == comp1) | (labels == comp2))[0]
# keep = np.where((labels == comp1))[0]
coords = coords_full[keep]
xy     = xy_full[keep]
y      = y_full[keep]
S      = y.shape[0]

# ================================================================
# Rebuild adjacency (merged)
# ================================================================
A = (squareform(pdist(xy)) <= 0.22).astype(int)
np.fill_diagonal(A, 0)
A = csr_matrix(A)

deg = np.array(A.sum(axis=1)).flatten()
Q_icar = diags(deg) - A
I_S    = diags(np.ones(S))

# ================================================================
# WEEKLY bins (1..52)
# ================================================================
time_bin = 52
# bin_edges = np.array([0, 13, 26, 39, 52])
bin_edges = np.arange(53)
# bin_edges = [1,52]
bin_str = "weekly_1_52"

# ================================================================
# Helpers
# ================================================================
def week_block_indices(w, K, time_bin, S):
    idx = np.empty(K * S, dtype=int)
    p = 0
    for k in range(K):
        start = (k * time_bin + w) * S
        idx[p:p+S] = np.arange(start, start + S)
        p += S
    return idx

def theta_slice(k, w, time_bin, S):
    start = (k * time_bin + w) * S
    return slice(start, start + S)

def log_lognormal(x, mu, sigma):
    if x <= 0:
        return -np.inf
    logx = np.log(x)
    return (
        -0.5 * ((logx - mu) / sigma) ** 2
        - np.log(x)
        - np.log(sigma)
        - 0.5 * np.log(2 * np.pi)
    )



# ================================================================
# ============================ p01 ================================
# ================================================================
loc_mask = (y[:, :-1] == 0)
row_idx, time_idx = np.where(loc_mask)
N = len(row_idx)

outcome = y[row_idx, time_idx + 1]
kappa = outcome - 0.5

t_raw   = time_idx + 1
t_trend = t_trend_full[time_idx]

week_in_year = ((t_raw - 1) % 52) + 1
week_raw = week_in_year - 1
# week_raw = ((week_in_year - 1) // 13)

cov = np.column_stack([
    np.ones(N), np.ones(N),
    np.cos(2*np.pi*week_in_year/period),
    np.cos(2*np.pi*week_in_year/period),
    np.sin(2*np.pi*week_in_year/period),
    np.sin(2*np.pi*week_in_year/period),
    t_trend, t_trend
])
K = cov.shape[1]

rows, cols, vals = [], [], []
for i in tqdm(range(N), desc="Building X | p01", leave=False):
    s = row_idx[i]
    w = week_raw[i]
    for k in range(K):
        col = s + S * (w + time_bin * k)
        rows.append(i)
        cols.append(col)
        vals.append(cov[i, k])

X = coo_matrix((vals, (rows, cols)),
               shape=(N, K * time_bin * S)).tocsr()

# ================================================================
# MCMC and Priors
# ================================================================
burn, thin, tot_save = 2000, 5, 1000
total_iters = burn + tot_save * thin

# ICAR hyper
a0_icar, b0_icar = 100.0, 10.0
log_sd_a0_icar, log_sd_b0_icar = 0.2, 0.2

# IID hyper
a0_iid, b0_iid = 10.0, 10.0
log_sd_a0_iid, log_sd_b0_iid = 0.05, 0.05

all_theta = np.zeros((K, time_bin, S, tot_save))
all_tau   = np.zeros((K, time_bin, tot_save))
all_a0_icar = np.zeros(tot_save)
all_b0_icar = np.zeros(tot_save)
all_a0_iid  = np.zeros(tot_save)
all_b0_iid  = np.zeros(tot_save)
mu_a = np.log(10)
sigma_a = 1

mu_b = np.log(1)
sigma_b = 1

icar_idx = [j for j in range(K*time_bin) if (j // time_bin) % 2 == 0]
iid_idx  = [j for j in range(K*time_bin) if (j // time_bin) % 2 == 1]

eps = 1e-5
# eps = 0
I_S = diags(np.ones(S)) 

Q_template = []
for k in range(K):
    if k % 2 == 0:
        Qk = Q_icar + eps * I_S 
    else:
        Qk = I_S
    for _ in range(time_bin):
        Q_template.append(Qk)

# Q_template is has first 52 blocks of ICAR, then 52 blocks of IID, then ....

curr_theta = np.zeros(K * time_bin * S)
curr_tau   = np.ones(K * time_bin) * 100.0
rhs = X.T @ kappa
save_idx = 0


In [19]:
# ---- 1. build prior precision (still block diag by k,w) ----
block_list = [curr_tau[j] * Q_template[j] for j in range(K*time_bin)]
curr_prec = bmat(
    [[block_list[i] if i == j else None for j in range(K*time_bin)]
    for i in range(K*time_bin)],
    format="csr"
)

In [ ]:
Q0 = Q_template[52].tocsr()

for i in range(Q0.shape[0]):
    start = Q0.indptr[i]
    end   = Q0.indptr[i+1]
    
    cols = Q0.indices[start:end]
    vals = Q0.data[start:end]
    
    for c, v in zip(cols, vals):
        print(f"row={i}, col={c}, value={v}")

row=0, col=0, value=1.0
row=1, col=1, value=1.0
row=2, col=2, value=1.0
row=3, col=3, value=1.0
row=4, col=4, value=1.0
row=5, col=5, value=1.0
row=6, col=6, value=1.0
row=7, col=7, value=1.0
row=8, col=8, value=1.0
row=9, col=9, value=1.0
row=10, col=10, value=1.0
row=11, col=11, value=1.0
row=12, col=12, value=1.0
row=13, col=13, value=1.0
row=14, col=14, value=1.0
row=15, col=15, value=1.0
row=16, col=16, value=1.0
row=17, col=17, value=1.0
row=18, col=18, value=1.0
row=19, col=19, value=1.0
row=20, col=20, value=1.0
row=21, col=21, value=1.0
row=22, col=22, value=1.0
row=23, col=23, value=1.0
row=24, col=24, value=1.0
row=25, col=25, value=1.0
row=26, col=26, value=1.0
row=27, col=27, value=1.0
row=28, col=28, value=1.0
row=29, col=29, value=1.0
row=30, col=30, value=1.0
row=31, col=31, value=1.0
row=32, col=32, value=1.0
row=33, col=33, value=1.0
row=34, col=34, value=1.0
row=35, col=35, value=1.0
row=36, col=36, value=1.0
row=37, col=37, value=1.0
row=38, col=38, value=1.0
row=39

In [ ]:

# ================================================================
# MCMC
# ================================================================
for it in tqdm(range(total_iters), desc="MCMC | p01", leave=True):

    # ---- 1. build prior precision (still block diag by k,w) ----
    block_list = [curr_tau[j] * Q_template[j] for j in range(K*time_bin)]
    curr_prec = bmat(
        [[block_list[i] if i == j else None for j in range(K*time_bin)]
        for i in range(K*time_bin)],
        format="csr"
    )

    # ---- 2. PG weights ----
    omega = random_polyagamma(1, X @ curr_theta)

    # ---- 3. full posterior precision (still block diag by week) ----
    post_prec = X.T.multiply(omega) @ X + curr_prec

    # ---- 4. block sampling by week ----
    for w in range(time_bin):

        idx = week_block_indices(w, K, time_bin, S)

        # extract block
        Q_w = post_prec[idx[:, None], idx]
        rhs_w = rhs[idx]

        try:
            factor = cholesky(Q_w, mode="simplicial")

            # posterior mean
            mu_w = factor.solve_A(rhs_w)

            # correct Gaussian draw
            z = np.random.randn(len(idx))
            z_perm = factor.apply_P(z)
            u = factor.solve_L(z_perm)
            noise_w = factor.apply_Pt(u)

        except:
            # fallback dense (block level only, safe)
            Q_w_dense = Q_w.toarray()

            mu_w = np.linalg.solve(Q_w_dense, rhs_w)

            L = np.linalg.cholesky(Q_w_dense)
            z = np.random.randn(len(idx))
            noise_w = np.linalg.solve(L, z)

        curr_theta[idx] = mu_w + noise_w




    for k in range(K):
        for w in range(time_bin):
            j = k*time_bin + w
            beta = curr_theta[j*S:(j+1)*S]
            Qj = Q_icar + eps * I_S  if (k % 2 == 0) else I_S
            rk = (S) if (k % 2 == 0) else S
            quad = beta @ (Qj @ beta)
            if k % 2 == 0:   # ICAR
                curr_tau[j] = np.random.gamma(
                    a0_icar + 0.5*rk,
                    1.0 / (b0_icar + 0.5*quad)
                )
            else:            # IID
                curr_tau[j] = np.random.gamma(
                    a0_iid + 0.5*rk,
                    1.0 / (b0_iid + 0.5*quad)
                )

    from scipy.special import gammaln

    tau_icar = curr_tau[icar_idx]
    tau_iid  = curr_tau[iid_idx]

    # =========================================================
    # ==================== ICAR hyper =========================
    # =========================================================

    # ---------- MH: a0_icar ----------
    log_a0_prop = np.log(a0_icar) + log_sd_a0_icar * np.random.randn()
    a0_prop = np.exp(log_a0_prop)

    ll_curr = np.sum(
        a0_icar*np.log(b0_icar)
        - gammaln(a0_icar)
        + (a0_icar-1)*np.log(tau_icar)
        - b0_icar*tau_icar
    )

    ll_prop = np.sum(
        a0_prop*np.log(b0_icar)
        - gammaln(a0_prop)
        + (a0_prop-1)*np.log(tau_icar)
        - b0_icar*tau_icar
    )

    logpost_curr = ll_curr + log_lognormal(a0_icar, mu_a, sigma_a)
    logpost_prop = (
        ll_prop
        + log_lognormal(a0_prop, mu_a, sigma_a)
        + log_a0_prop
        - np.log(a0_icar)
    )

    if np.log(np.random.rand()) < (logpost_prop - logpost_curr):
        a0_icar = a0_prop


    # ---------- MH: b0_icar ----------
    log_b0_prop = np.log(b0_icar) + log_sd_b0_icar * np.random.randn()
    b0_prop = np.exp(log_b0_prop)

    ll_curr = np.sum(
        a0_icar*np.log(b0_icar)
        - gammaln(a0_icar)
        + (a0_icar-1)*np.log(tau_icar)
        - b0_icar*tau_icar
    )

    ll_prop = np.sum(
        a0_icar*np.log(b0_prop)
        - gammaln(a0_icar)
        + (a0_icar-1)*np.log(tau_icar)
        - b0_prop*tau_icar
    )

    logpost_curr = ll_curr + log_lognormal(b0_icar, mu_b, sigma_b)
    logpost_prop = (
        ll_prop
        + log_lognormal(b0_prop, mu_b, sigma_b)
        + log_b0_prop
        - np.log(b0_icar)
    )

    if np.log(np.random.rand()) < (logpost_prop - logpost_curr):
        b0_icar = b0_prop


    # =========================================================
    # ===================== IID hyper =========================
    # =========================================================

    # ---------- MH: a0_iid ----------
    log_a0_prop = np.log(a0_iid) + log_sd_a0_iid * np.random.randn()
    a0_prop = np.exp(log_a0_prop)

    ll_curr = np.sum(
        a0_iid*np.log(b0_iid)
        - gammaln(a0_iid)
        + (a0_iid-1)*np.log(tau_iid)
        - b0_iid*tau_iid
    )

    ll_prop = np.sum(
        a0_prop*np.log(b0_iid)
        - gammaln(a0_prop)
        + (a0_prop-1)*np.log(tau_iid)
        - b0_iid*tau_iid
    )

    logpost_curr = ll_curr + log_lognormal(a0_iid, mu_a, sigma_a)
    logpost_prop = (
        ll_prop
        + log_lognormal(a0_prop, mu_a, sigma_a)
        + log_a0_prop
        - np.log(a0_iid)
    )

    if np.log(np.random.rand()) < (logpost_prop - logpost_curr):
        a0_iid = a0_prop


    # ---------- MH: b0_iid ----------
    log_b0_prop = np.log(b0_iid) + log_sd_b0_iid * np.random.randn()
    b0_prop = np.exp(log_b0_prop)

    ll_curr = np.sum(
        a0_iid*np.log(b0_iid)
        - gammaln(a0_iid)
        + (a0_iid-1)*np.log(tau_iid)
        - b0_iid*tau_iid
    )

    ll_prop = np.sum(
        a0_iid*np.log(b0_prop)
        - gammaln(a0_iid)
        + (a0_iid-1)*np.log(tau_iid)
        - b0_prop*tau_iid
    )

    logpost_curr = ll_curr + log_lognormal(b0_iid, mu_b, sigma_b)
    logpost_prop = (
        ll_prop
        + log_lognormal(b0_prop, mu_b, sigma_b)
        + log_b0_prop
        - np.log(b0_iid)
    )

    if np.log(np.random.rand()) < (logpost_prop - logpost_curr):
        b0_iid = b0_prop


    
    clear_output(wait=True)
    print(f"a0 = {a0_icar:.4f}, b0 = {b0_icar:.4f}")
    print(f"a0 = {a0_iid:.4f}, b0 = {b0_iid:.4f}")
    tau_mat = curr_tau.reshape(K, time_bin)
    display(pd.DataFrame(tau_mat.T))
    if it >= burn and (it-burn) % thin == 0:
        for k in range(K):
            for w in range(time_bin):
                all_theta[k,w,:,save_idx] = curr_theta[theta_slice(k,w,time_bin,S)]
        all_tau[:,:,save_idx] = tau_mat
        all_a0_icar[save_idx] = a0_icar
        all_b0_icar[save_idx] = b0_icar
        all_a0_iid[save_idx] = a0_iid
        all_b0_iid[save_idx] = b0_iid
        save_idx += 1
        if save_idx == tot_save:
            break

with open(f"BYM_{bin_str}_p01.pkl", "wb") as f:
    pickle.dump({
        "keep_idx": keep,
        "theta": all_theta,
        "tau": all_tau,
        "a0_icar": all_a0_icar,
        "b0_icar": all_b0_icar,
        "a0_iid": all_a0_iid,
        "b0_iid": all_b0_iid,
        "bin_edges": bin_edges
    }, f)
